In [ ]:
# THIS CELL FOR GETTING SBIR AWARDS DATA

import requests
import sqlite3
import time
import json

# API base URL
BASE_URL = "https://api.www.sbir.gov/public/api/awards"

# Database setup
DB_NAME = "server/awards.db"

# Function to create the database tables
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Create awards table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS awards (
            award_link INTEGER PRIMARY KEY,
            firm TEXT,
            award_title TEXT,
            agency TEXT,
            branch TEXT,
            phase TEXT,
            program TEXT,
            agency_tracking_number TEXT,
            contract TEXT,
            proposal_award_date TEXT,
            contract_end_date TEXT,
            solicitation_number TEXT,
            solicitation_year INTEGER,
            topic_code TEXT,
            award_year INTEGER,
            award_amount DECIMAL(15,4),
            duns TEXT,
            uei TEXT,
            hubzone_owned TEXT,
            socially_economically_disadvantaged TEXT,
            women_owned TEXT,
            number_employees INTEGER,
            company_url TEXT,
            address1 TEXT,
            address2 TEXT,
            city TEXT,
            state TEXT,
            zip TEXT,
            poc_name TEXT,
            poc_title TEXT,
            poc_phone TEXT,
            poc_email TEXT,
            pi_name TEXT,
            pi_title TEXT,
            pi_phone TEXT,
            pi_email TEXT,
            ri_name TEXT,
            ri_poc_name TEXT,
            ri_poc_phone TEXT,
            research_area_keywords TEXT,
            abstract TEXT
        )
    """)

    conn.commit()
    conn.close()

# Function to fetch data from API
def fetch_data(page):
    params = {
        "rows": 50,
        "start": page * 50,
        #"firm": "xyz",
        #"agency": "HHS",
        #'year': 2024,
        #'ri': "Stanford",
        }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching page {page}: {response.status_code}")
        return None

# Function to insert data into SQLite database
def insert_data(data):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in data:
        cursor.execute("""
            INSERT OR REPLACE INTO awards (
                award_link, firm, award_title, agency, branch, phase, program,
                agency_tracking_number, contract, proposal_award_date, contract_end_date,
                solicitation_number, solicitation_year, topic_code, award_year, award_amount,
                duns, uei, hubzone_owned, socially_economically_disadvantaged, women_owned,
                number_employees, company_url, address1, address2, city, state, zip,
                poc_name, poc_title, poc_phone, poc_email,
                pi_name, pi_title, pi_phone, pi_email,
                ri_name, ri_poc_name, ri_poc_phone,
                research_area_keywords, abstract
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            item.get("award_link"),
            item.get("firm"),
            item.get("award_title"),
            item.get("agency"),
            item.get("branch"),
            item.get("phase"),
            item.get("program"),
            item.get("agency_tracking_number"),
            item.get("contract"),
            item.get("proposal_award_date"),
            item.get("contract_end_date"),
            item.get("solicitation_number"),
            item.get("solicitation_year"),
            item.get("topic_code"),
            item.get("award_year"),
            item.get("award_amount"),
            item.get("duns"),
            item.get("uei"),
            item.get("hubzone_owned"),
            item.get("socially_economically_disadvantaged"),
            item.get("women_owned"),
            item.get("number_employees"),
            item.get("company_url"),
            item.get("address1"),
            item.get("address2"),
            item.get("city"),
            item.get("state"),
            item.get("zip"),
            item.get("poc_name"),
            item.get("poc_title"),
            item.get("poc_phone"),
            item.get("poc_email"),
            item.get("pi_name"),
            item.get("pi_title"),
            item.get("pi_phone"),
            item.get("pi_email"),
            item.get("ri_name"),
            item.get("ri_poc_name"),
            item.get("ri_poc_phone"),
            item.get("research_area_keywords"),
            item.get("abstract")
        ))    
    conn.commit()
    conn.close()

# Main execution
def main():
    setup_database()
    
    #for page in range(3):  #number of pages
    page = 0 #around 4300 total pages
    while True:
        print(f"Fetching page {page + 1}...")
        data = fetch_data(page)
        if not data or len(data) == 0:
            print("No more data")
            break
        insert_data(data)
        time.sleep(0.1)  # Avoid excessive requests
        page += 1

    print("Awards data successfully stored in SQLite database.")

if __name__ == "__main__":
    main()


In [ ]:
def check_schema():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='awards'")
    print("\nAwards table schema:")
    print(cursor.fetchone()[0])
    
    conn.close()

def fetch_awards():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM awards") # LIMIT 10")
    rows = cursor.fetchall()
    
    for row in rows:
        print(row)
    
    conn.close()

def count_entries():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Count entries in awards table
    cursor.execute("SELECT COUNT(*) FROM awards")
    awards_count = cursor.fetchone()[0]
    
    conn.close()
    
    return {
        "awards": awards_count,
    }

if __name__ == "__main__":
    check_schema()
    
    print("Awards:")
    fetch_awards()

    counts = count_entries()
    print(f"Number of awards: {counts['awards']}")
